# Preprocessing and leakage-safe pipelines

Learn how to split data before fitting transformations, impute missing values, encode categories, and keep preprocessing with the classifier. This lesson generates a small teaching dataset locally; its scores are not estimates for a real application.

**Prerequisites:** arrays, DataFrames, and basic classification. **Environment:** the repository's base requirements. **Execution:** restart the kernel and run all cells in order. No files, downloads, or GPU are required.


In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import balanced_accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
rng = np.random.default_rng(SEED)
n = 360
numeric = rng.normal(size=(n, 2))
channel = rng.choice(['web', 'store', 'phone'], size=n)
logit = 1.5 * numeric[:, 0] - 0.8 * numeric[:, 1] + (channel == 'web') * 0.7
y = (logit + rng.normal(0, 0.8, n) > 0).astype(int)
X = pd.DataFrame({'feature_a': numeric[:, 0], 'feature_b': numeric[:, 1], 'channel': channel})
X.loc[rng.choice(n, 30, replace=False), 'feature_a'] = np.nan
X.loc[rng.choice(n, 25, replace=False), 'channel'] = np.nan
display(X.head())
display(X.isna().sum().to_frame('missing'))


## 1. Hold out the test set first

The imputer's median, scaler statistics, and category vocabulary must come from training data. Splitting first keeps the test set out of those decisions. `stratify=y` retains approximately the same class proportions in each split.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)
print('Train:', X_train.shape, 'Test:', X_test.shape)


## 2. Build separate numeric and categorical transformations

Unknown categories at inference are allowed with `handle_unknown='ignore'`. The transformations sit inside the estimator pipeline, so future cross-validation also fits them within each training fold.


In [ ]:
numeric_steps = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('scale', StandardScaler()),
])
category_steps = Pipeline([
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('encode', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])
prepare = ColumnTransformer([
    ('numeric', numeric_steps, ['feature_a', 'feature_b']),
    ('category', category_steps, ['channel']),
])
model = Pipeline([
    ('prepare', prepare),
    ('classify', LogisticRegression(max_iter=500, random_state=SEED)),
])
model.fit(X_train, y_train)
prediction = model.predict(X_test)
print(classification_report(y_test, prediction, zero_division=0))
print('Balanced accuracy:', round(balanced_accuracy_score(y_test, prediction), 3))


## 3. Inspect the fitted transformations

This check compares the fitted imputation statistic to the training median, independently of the estimator's predictions. The test data is transformed with the existing training statistics.


In [ ]:
fitted = model.named_steps['prepare']
medians = fitted.named_transformers_['numeric'].named_steps['impute'].statistics_
np.testing.assert_allclose(medians, X_train[['feature_a', 'feature_b']].median().to_numpy())
print('Training medians:', medians)
print('Output columns:', fitted.get_feature_names_out().tolist())

new_rows = pd.DataFrame({'feature_a': [0.2, np.nan], 'feature_b': [-0.3, 0.8],
                         'channel': ['new_channel', 'store']})
print('New-row predictions:', model.predict(new_rows))
assert np.isfinite(fitted.transform(new_rows)).all()


## Practice

Compare median and mean imputation using training-only cross-validation. Add another category and verify that the pipeline still accepts it. Do not repeatedly choose settings from the test score.

**Reference:** [scikit-learn: pipelines and leakage](https://scikit-learn.org/stable/common_pitfalls.html).
